# Day 2 · 실습 ⑥. Patch Embedding과 Attention 시각화

**연계 이론:** 이론 3. Vision Transformer의 구조 · 이론 4. Self-Attention의 계산 원리

## 학습 목표

1. 224×224 이미지를 16×16과 32×32 Patch로 나누어 비교합니다.
2. Patch Flatten, Linear Projection, CLS Token, Positional Embedding의 Shape을 확인합니다.
3. Scaled Dot-Product Attention을 계산하고 Heatmap으로 시각화합니다.

> 이 Notebook은 학습 원리 확인용입니다. 생성한 Embedding 가중치는 **무작위 초기화**이며 사전학습 ViT의 출력이 아닙니다. 실제 사전학습 ResNet18·ViT-B/16 비교는 실습 ⑦에서 진행합니다.

**진행 방법:** 셀을 위에서 아래로 실행하고, `# 변경해 보기`가 표시된 값을 바꾸어 결과를 확인합니다.


## 1. 이미지와 Tensor 준비

외부에서 이미지를 다운로드하지 않아도 진행할 수 있도록 도형 이미지를 만듭니다. 개인 이미지를 쓰려면 `IMAGE_PATH`에 파일 경로를 입력합니다.


In [ ]:
from pathlib import Path
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
IMAGE_SIZE = 224
IMAGE_PATH = ""  # 변경해 보기: "sample.jpg" 또는 비워 두기

if IMAGE_PATH.strip():
    path = Path(IMAGE_PATH)
    if not path.is_file():
        raise FileNotFoundError(f"이미지 파일을 확인하세요: {path}")
    image = Image.open(path).convert("RGB").resize((224, 224))
else:
    image = Image.new("RGB", (224, 224), (242, 247, 243))
    draw = ImageDraw.Draw(image)
    draw.rectangle((15, 20, 105, 112), fill=(222, 87, 76))
    draw.ellipse((115, 25, 210, 120), fill=(49, 156, 109))
    draw.polygon([(20, 205), (110, 130), (195, 205)], fill=(62, 118, 207))

image_tensor = (torch.from_numpy(np.asarray(image).copy())
                .permute(2, 0, 1).float().div(255).unsqueeze(0))
print("PyTorch:", torch.__version__)
print("이미지 Shape (N, C, H, W):", tuple(image_tensor.shape))
assert image_tensor.shape == (1, 3, 224, 224)
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title("Input image (224 x 224)")
plt.axis("off")
plt.show()


## 2. Patch Size에 따른 이미지 분할

224×224 이미지에서 Patch Size가 16이면 14×14=196개, 32이면 7×7=49개입니다. Patch는 이미지의 의미가 아니라 정해진 격자로 구분합니다.


In [ ]:
PATCH_SIZES = [16, 32]  # 변경해 보기: [8, 16, 32]
fig, axes = plt.subplots(1, len(PATCH_SIZES), figsize=(5 * len(PATCH_SIZES), 5))
axes = np.atleast_1d(axes)
for ax, patch_size in zip(axes, PATCH_SIZES):
    if IMAGE_SIZE % patch_size:
        raise ValueError("Patch Size는 224를 나눌 수 있어야 합니다.")
    n_side = IMAGE_SIZE // patch_size
    ax.imshow(image)
    for pos in range(0, IMAGE_SIZE + 1, patch_size):
        ax.axhline(pos - 0.5, color="green", linewidth=0.7)
        ax.axvline(pos - 0.5, color="green", linewidth=0.7)
    ax.set_title(f"Patch {patch_size} x {patch_size}: {n_side**2} patches")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("16x16:", (224 // 16)**2, "개 / 32x32:", (224 // 32)**2, "개")


## 3. Patch Flatten

각 Patch에 포함된 RGB 픽셀값을 1차원으로 펼칩니다. 16×16 Patch의 Flatten 차원은 3×16×16=768이고, 32×32 Patch는 3×32×32=3072입니다.


In [ ]:
def extract_patches(x, patch_size):
    # (N, 3, 224, 224) → (N, Patch 수, 3 * Patch Size**2)
    return F.unfold(x, kernel_size=patch_size, stride=patch_size).transpose(1, 2)

flat16 = extract_patches(image_tensor, 16)
flat32 = extract_patches(image_tensor, 32)
print("Patch 16 Flatten:", tuple(flat16.shape))
print("Patch 32 Flatten:", tuple(flat32.shape))
assert flat16.shape == (1, 196, 768)
assert flat32.shape == (1, 49, 3072)


In [ ]:
# 실제 이미지에서 잘라낸 Patch를 확인합니다.
SELECTED_PATCHES = [0, 6, 13, 91, 98, 182]  # 변경해 보기: 0~195
fig, axes = plt.subplots(1, len(SELECTED_PATCHES), figsize=(12, 2.5))
for ax, idx in zip(axes, SELECTED_PATCHES):
    patch = flat16[0, idx].reshape(3, 16, 16).permute(1, 2, 0)
    ax.imshow(patch.numpy())
    ax.set_title(f"Patch {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Patch Embedding과 CLS Token

Flatten은 픽셀값을 펼치는 연산이고, Linear Projection은 학습 가능한 가중치로 새로운 벡터를 만드는 연산입니다. 이 예제에서는 출력 Embedding 차원을 768로 맞춥니다.


In [ ]:
EMBED_DIM = 768
projection16 = nn.Linear(3 * 16 * 16, EMBED_DIM)
projection32 = nn.Linear(3 * 32 * 32, EMBED_DIM)
with torch.no_grad():
    embedding16 = projection16(flat16)
    embedding32 = projection32(flat32)
print("Patch 16 Embedding:", tuple(embedding16.shape))
print("Patch 32 Embedding:", tuple(embedding32.shape))
assert embedding16.shape == (1, 196, 768)
assert embedding32.shape == (1, 49, 768)
print("주의: 무작위 초기화된 Projection이며 학습된 특징이 아닙니다.")


In [ ]:
def add_cls_and_position(embeddings):
    batch, token_count, dim = embeddings.shape
    cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
    pos_embedding = nn.Parameter(torch.randn(1, token_count + 1, dim) * 0.02)
    cls_batch = cls_token.expand(batch, -1, -1)
    tokens = torch.cat([cls_batch, embeddings], dim=1)
    return tokens + pos_embedding

encoder16 = add_cls_and_position(embedding16)
encoder32 = add_cls_and_position(embedding32)
print("CLS + Position, Patch 16:", tuple(encoder16.shape))
print("CLS + Position, Patch 32:", tuple(encoder32.shape))
assert encoder16.shape == (1, 197, 768)
assert encoder32.shape == (1, 50, 768)


## 5. Q·K·V와 Scaled Dot-Product Attention

이론 4의 **두 Token 계산 예제**를 재현합니다. Q·K·V는 계산 원리를 보여주기 위해 고정한 값입니다.

| Token | Q | K | V |
|---|---|---|---|
| A | [1, 0] | [1, 0] | [2, 0] |
| B | [0, 1] | [0, 1] | [0, 4] |

계산 순서: $QK^T$ → $\div\sqrt{d_k}$ → Softmax → Weight × Value


In [ ]:
Q = torch.tensor([[1., 0.], [0., 1.]])
K = torch.tensor([[1., 0.], [0., 1.]])
V = torch.tensor([[2., 0.], [0., 4.]])

scores = Q @ K.T
scaled_scores = scores / math.sqrt(Q.shape[-1])
weights = torch.softmax(scaled_scores, dim=-1)
outputs = weights @ V

print("1. 내적 Score:\n", scores)
print("2. Scaling 후:\n", scaled_scores)
print("3. Softmax 가중치:\n", weights)
print("4. 가중합 출력:\n", outputs)
print("각 행의 합:", weights.sum(dim=-1))
assert torch.allclose(weights[0], torch.tensor([.670, .330]), atol=.002)
assert torch.allclose(outputs[0], torch.tensor([1.34, 1.32]), atol=.01)


## 6. Attention Heatmap

Heatmap의 **행은 Query**, **열은 Key**, 색의 농도는 Attention Weight를 나타냅니다. 각 Query 행의 가중치 합은 1입니다.


In [ ]:
heat = weights.detach().numpy()
fig, ax = plt.subplots(figsize=(5.4, 4.5))
im = ax.imshow(heat, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks([0, 1], ["Token A", "Token B"])
ax.set_yticks([0, 1], ["Token A", "Token B"])
ax.set_xlabel("Key")
ax.set_ylabel("Query")
for r in range(2):
    for c in range(2):
        ax.text(c, r, f"{heat[r,c]:.3f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.show()


### 6.1 세 Token으로 확장하여 Query별 분포 비교

`QUERY_INDEX`를 바꾸면 어떤 Query 행을 자세히 관찰할지 변경할 수 있습니다. 여기서도 학습 모델이 아닌 계산 예제를 사용합니다.


In [ ]:
labels = ["Patch A", "Patch B", "Patch C"]
Q3 = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])
K3 = Q3.clone()
V3 = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
weights3 = torch.softmax((Q3 @ K3.T) / math.sqrt(2), dim=-1)
outputs3 = weights3 @ V3
QUERY_INDEX = 0  # 변경해 보기: 0, 1, 2
assert QUERY_INDEX in (0, 1, 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
mat = weights3.numpy()
im = ax1.imshow(mat, cmap="Greens", vmin=0, vmax=1)
ax1.set_xticks(range(3), labels)
ax1.set_yticks(range(3), labels)
ax1.set_xlabel("Key")
ax1.set_ylabel("Query")
for r in range(3):
    for c in range(3):
        ax1.text(c, r, f"{mat[r,c]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax1)
ax2.bar(labels, mat[QUERY_INDEX], color="seagreen")
ax2.set_ylim(0, 1)
ax2.set_title(f"Query: {labels[QUERY_INDEX]}")
ax2.set_ylabel("Attention weight")
plt.tight_layout()
plt.show()
print("가중합 결과:", outputs3[QUERY_INDEX])


### 6.2 Score 크기를 변경하여 Softmax 비교

Score에 곱하는 값을 바꾸어 가중치 분포가 어떻게 변하는지 비교합니다. 이 값은 실제 사전학습 모델의 설정값이 아니라 교육용 조건입니다.


In [ ]:
MULTIPLIERS = [0.5, 1.0, 2.0]  # 변경해 보기: [0.25, 1.0, 4.0]
base_scores = (Q3 @ K3.T) / math.sqrt(2)
fig, axes = plt.subplots(1, len(MULTIPLIERS), figsize=(4 * len(MULTIPLIERS), 4))
axes = np.atleast_1d(axes)
for ax, multiplier in zip(axes, MULTIPLIERS):
    current = torch.softmax(base_scores * multiplier, dim=-1).numpy()
    ax.imshow(current, cmap="Greens", vmin=0, vmax=1)
    ax.set_title(f"Score x {multiplier}")
    ax.set_xticks(range(3), labels)
    ax.set_yticks(range(3), labels)
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f"{current[r,c]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()


## 7. Patch Size와 Attention Matrix 크기

CLS Token을 포함하면 Patch Size 16은 197×197, Patch Size 32는 50×50의 Attention Matrix를 만듭니다(Head 하나 기준). Patch Size를 바꾼다고 사전학습된 ViT-B/16의 가중치를 그대로 쓸 수 있는 것은 아닙니다.


In [ ]:
for p in [16, 32]:
    patch_count = (224 // p)**2
    token_count = patch_count + 1  # CLS Token 포함
    print(f"Patch {p}: patches={patch_count}, tokens={token_count}, "
          f"attention shape=({token_count}, {token_count}), "
          f"entries={token_count**2:,}")
assert (196 + 1)**2 == 38809
assert (49 + 1)**2 == 2500


## 8. 실습 결과 정리

| 확인 항목 | Patch 16 | Patch 32 |
|---|---:|---:|
| 이미지 Patch 개수 | 196 | 49 |
| Flatten된 Patch 하나의 픽셀값 수 | 768 | 3072 |
| Embedding 차원(본 실습) | 768 | 768 |
| CLS를 포함한 Token 수 | 197 | 50 |
| Attention 행렬의 크기(Head 하나) | 197×197 | 50×50 |

**핵심 질문**

1. Patch Size를 16에서 32로 늘리면 Token 수는 어떻게 달라지나요?
2. Flatten 차원과 Embedding 차원의 차이는 무엇인가요?
3. CLS Token이 추가되면 Token 수가 하나 증가하는 이유는 무엇인가요?
4. Attention Matrix의 행과 열은 각각 무엇을 의미하나요?
5. Query와 Key의 역할, Value의 역할을 구분할 수 있나요?

**다음 실습:** 실습 ⑦에서 실제 사전학습 ResNet18·ViT-B/16의 추론 결과와 중간 표현을 비교합니다.
